## Instructions

Before running this notebook, make sure the following files are located in the same directory:

- `Case1.ipynb`
- `inforce_policies_aggregated_0.5.0.csv`

Run the cells in `Case1.ipynb` to perform the analysis using the provided dataset.

In [ ]:
# Uncomment the following line to install required packages if needed:
# !pip install pandas numpy scikit-learn lightgbm optuna plotnine shap tqdm -q

In [ ]:
# Utility
from types import MethodType
from tqdm.auto import tqdm

# Data Manipulation
import pandas as pd
import numpy as np

# Predictive Modeling
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_poisson_deviance, mean_squared_error
from lightgbm import LGBMRegressor

# Optimization
import optuna

# Visualization
from plotnine import *

# Model Interpretation
from sklearn.inspection import permutation_importance, partial_dependence
import shap

In [ ]:
class CONFIG:
    RANDOM_STATE = 0
    N_TRIALS_A = 0

In [ ]:
import os

# Define the file name
file_name = 'inforce_policies_aggregated_0.5.0.csv'

# Load the CSV file into data_df
if os.path.exists(file_name):
    data_df = pd.read_csv(file_name)
    print(f"Loaded {file_name} successfully.")
else:
    print(f"File not found: {file_name}")
    print("Please check if the file exists in the current working directory.")

In [ ]:
data_df.head()

In [ ]:
# Convert age bands to numeric values
data_df['attained_age_band_num'] = data_df['attained_age_band'].map({
    '15-19':17.5,
    '20-24':22.5,
    '25-29':27.5,
    '30-34':32.5,
    '35-39':37.5,
    '40-44':42.5,
    '45-49':47.5,
    '50-54':52.5,
    '55-59':57.5,
    '60-64':62.5,
    '65-69':67.5,
    '70-74':72.5,
    '75-79':77.5,
    '80-84':82.5,
    '85-89':87.5,
    '90-94':92.5,
    '95-99':97.5,
    '100-104':102.5,
    '105-109':107.5,
}).to_numpy()

# Convert duration-year bands to numeric values
data_df['duration_year_band_num'] = data_df['duration_year_band'].map({
    '0':0,
    '1':1,
    '2':2,
    '3':3,
    '4':4,
    '5':5,
    '6':6,
    '7':7,
    '8':8,
    '9':9,
    '10':10,
    '11':11,
    '12':12,
    '13':13,
    '14':14,
    '15+':15
}).to_numpy()

In [ ]:
data_df.info()

In [ ]:
# Features
categorical_features = [
    'gender',
    'conversion_indicator',
    'rider_attachment_timing',
    'renewal_status',
    'sum_assured_band',
    'base_product_category',
    'corporate_indicator',
    'underwriting_method',
    'distribution_channel',
    'sales_prefecture_group',
]

numerical_features = [
    'attained_age_band_num',
    'duration_year_band_num',
]

# Convert Data Types
for col in categorical_features:
    if col in data_df.columns:
        data_df[col] = data_df[col].astype('category')

# Restore explicit ordering for ordinal categorical features
if 'sum_assured_band' in data_df.columns:
    data_df['sum_assured_band'] = pd.Categorical(
        data_df['sum_assured_band'],
        categories=['1-10', '11-20', '21-30', '31-40'],
        ordered=True
    )

for col in numerical_features:
    if col in data_df.columns:
        data_df[col] = data_df[col].astype(float)

# Calculate Targets
data_df['p_index'] = data_df['actual_claim_amount'] / data_df['expected_claim_amount']

data_df.head()

In [ ]:
# Split Data
train_df, test_df = train_test_split(
    data_df,
    test_size=0.2,
    random_state=CONFIG.RANDOM_STATE
)

all_features = numerical_features + categorical_features

X_train = train_df[all_features]
X_test  = test_df[all_features]

## Model Fitting

In [ ]:
def set_clipped_predict(model):
    if not hasattr(model, '_original_predict'):
        model._original_predict = model.predict
    def clipped_predict(self, X, **kwargs):
        preds = self._original_predict(X, **kwargs)
        return np.clip(preds, 1e-6, None)
    clipped_predict.__name__ = "predict"
    model.predict = MethodType(clipped_predict, model)
    return model

In [ ]:
# APPROACH : Squared error loss, direct modeling of the A/E ratio

y_train = train_df['p_index']
w_train = train_df['expected_claim_amount']

y_test  = test_df['p_index']
w_test  = test_df['expected_claim_amount']

# Hyperparameter tuning
def objective(trial):
    params = {
        "objective": "regression",
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 5),
        "min_child_samples": trial.suggest_int("min_child_samples", 100, 2000),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "random_state": 42,
        "verbose": -1
    }
    model = LGBMRegressor(**params)
    model.fit(
        X=X_train,
        y=y_train,
        sample_weight=w_train,
        categorical_feature=categorical_features
    )
    model = set_clipped_predict(model)
    preds = model.predict(X_test)
    loss = mean_squared_error(y_test, preds, sample_weight=w_test)
    return loss

params = {
    'learning_rate': 0.008388289348989848,
    'n_estimators': 795,
    'max_depth': 3,
    'min_child_samples': 525,
    'reg_lambda': 3.6258609270887616,
    'reg_alpha': 0.0010586009698546866
}

if CONFIG.N_TRIALS_A:
  study = optuna.create_study(direction="minimize")
  study.optimize(objective, n_trials=CONFIG.N_TRIALS_A)
  params.update(study.best_params)
  print("Best deviance:", study.best_value)
  print("Best parameters:", study.best_params)

# Build model_WLS using the tuned hyperparameters
model_WLS = LGBMRegressor(
    **params,
    objective='regression',
    random_state=42,
    verbose=-1
)
model_WLS.fit(
    X=X_train,
    y=y_train,
    sample_weight=w_train,
    categorical_feature=categorical_features
)
model_WLS = set_clipped_predict(model_WLS)

# Predict the A/E ratio
pred_index_WLS = model_WLS.predict(X_test)
print(f"Average A/E ratio: {np.average(pred_index_WLS, weights=test_df['expected_claim_amount']):.4f}")

## Model Evaluation

In [ ]:
# Baseline for the null model (assuming all policies have the overall average A/E ratio)
base_index = np.average(train_df['p_index'], weights=train_df['expected_claim_amount'])
assert train_df['actual_claim_amount'].sum() / train_df['expected_claim_amount'].sum() == base_index
null_pred_indices = np.full_like(test_df['p_index'], base_index)

# Calculate the Poisson deviance (target = actual A/E ratio, weight = expected claim amount)
null_deviance = mean_poisson_deviance(test_df['p_index'], null_pred_indices, sample_weight=test_df['expected_claim_amount'])
deviance_WLS = mean_poisson_deviance(
    test_df['p_index'],
    pred_index_WLS,
    sample_weight=test_df['expected_claim_amount']
)

print(f"Null Model Poisson Deviance  : {null_deviance:.4f}")
print(f"WLS Approach Poisson Deviance : {deviance_WLS:.4f} (Pseudo-R2: {1 - (deviance_WLS / null_deviance):.4%})")

## Interpretation

In [ ]:
model = model_WLS # model_WLS allows TreeSHAP values to be interpreted as additive contributions to the A/E ratio
pred_index = pred_index_WLS

In [ ]:
FEATURE_DISPLAY_MAP = {
    'duration_year_band_num': 'Duration Year',
    'sum_assured_band': 'Sum Assured',
    'attained_age_band_num': 'Attained Age',
    'base_product_category': 'Base Product',
    'gender': 'Gender',
    'sales_prefecture_group': 'Sales Prefecture',
    'distribution_channel': 'Distribution Channel',
    'underwriting_method': 'Underwriting Method',
    'renewal_status': 'Renewal Status',
    'rider_attachment_timing': 'Rider Attachment Timing',
    'conversion_indicator': 'Conversion Indicator',
    'corporate_indicator': 'Corporate Indicator',
}

### Permutation Feature Importance

In [ ]:
# calculate PFI
pfi_results = permutation_importance(
    estimator=model,
    X=X_test,
    y=test_df['p_index'],
    sample_weight=test_df['expected_claim_amount'],
    scoring=["neg_mean_squared_error"], # ["neg_mean_poisson_deviance", "r2"],
    random_state=CONFIG.RANDOM_STATE,
    n_repeats=20
)

# construct a DataFrame for plotting
scoring_measure = 'neg_mean_squared_error'
pfi_df = pd.DataFrame({
    'feature': all_features,
    'importance_mean': pfi_results[scoring_measure].importances_mean,
    'importance_std': pfi_results[scoring_measure].importances_std
}).sort_values('importance_mean', ascending=True)
pfi_df['feature'] = pfi_df['feature'].map(FEATURE_DISPLAY_MAP)
pfi_df['feature'] = pd.Categorical(pfi_df['feature'], categories=pfi_df['feature'])

# create the plot with error bars
pfi_plot = (
    ggplot(pfi_df) +
    geom_col(aes(x='feature', y='importance_mean'), fill='#4477aa') +
    geom_errorbar(
        aes(x='feature', ymin='importance_mean - importance_std', ymax='importance_mean + importance_std'),
        width=0.2, color='#333333'
    ) +
    coord_flip() +
    theme_minimal() +
    labs(
        # title='Permutation Feature Importance',
        y='Increase in Mean Squared Error after Permutation',
        x='',
    ) +
    theme(figure_size=(8, 3))
)

print("\n-- PFI --\n")
pfi_plot

### Partial Dependence

In [ ]:
for feature in all_features:
    print(f"\n--- {feature} ---\n")

    # Calculate grouped A/E ratios by feature (Traditional Analysis Model)
    agg_df = train_df.groupby(feature, observed=False).agg(
        total_actual=('actual_claim_amount', 'sum'),
        total_expected=('expected_claim_amount', 'sum')
    ).reset_index()
    agg_df['p_index'] = agg_df['total_actual'] / agg_df['total_expected']
    agg_df['method'] = 'Grouped A/E (Traditional Analysis)'

    # Calculate partial dependence plot (PDP)
    pdp_results = partial_dependence(
        estimator=model,
        X=X_train,
        features=[feature],
        sample_weight=train_df['expected_claim_amount'],
        kind='average'
    )
    pdp_df = pd.DataFrame({
        feature: pdp_results['grid_values'][0],
        'p_index': pdp_results['average'][0],
        'method': 'PDP'
    })

    # Create the plot
    p = ggplot(
        data=pd.concat([agg_df, pdp_df]),
        mapping=aes(x=feature, y='p_index', color='method', group='method', linetype='method')
        ) + \
        geom_line(size=1) + \
        geom_point(size=2) + \
        labs(
            # title='Partial Dependence Plot',
            #subtitle=f"Comparison: PDP vs Grouped A/E for {feature}",
            x=FEATURE_DISPLAY_MAP[feature],
            y='',
            color='',
            linetype=''
        ) + \
        scale_linetype_manual(values=['--', '-']) + \
        scale_color_manual(values=['#ddaa33', '#004488']) + \
        theme_minimal() + \
        theme(legend_position="bottom", figure_size=(5, 3))

    if feature in categorical_features:
        p += scale_x_discrete(expand=(0.1, 0.1))

    p.show()

### Friedman's H

In [ ]:
def compute_pd_table(model, X, features, sample_weights=None):

    def _get_grid(X, features, sample_weights=None):
        df = X.copy()
        df['__weights'] = 1
        if sample_weights is not None:
            df['__weights'] = sample_weights
        grid = (
            df.groupby(features, observed=False)
            .agg(pd_weights=('__weights', 'sum'))
            .reset_index()
        )
        return grid

    def _get_pd_value(model, X, features, values, sample_weights=None):
        df = X.copy()
        original_dtypes = X.dtypes
        df[features] = values
        df = df.astype(original_dtypes)
        preds = model.predict(df)
        return np.average(preds, weights=sample_weights)

    grid_df = _get_grid(X, features, sample_weights)
    pd_values = []
    for values in tqdm(grid_df[features].values):
        pd_value = _get_pd_value(model, X, features, values, sample_weights)
        pd_values.append(pd_value)

    pd_center = np.average(pd_values, weights=grid_df['pd_weights'].values)
    grid_df['pd_values'] = pd_values
    grid_df['pd_centered'] = pd_values - pd_center

    return grid_df

In [ ]:
import itertools

print("--- 1D PDP precomputation ---")
pdp_1d_cache = {}

X_hstat = X_train.sample(2000, random_state = CONFIG.RANDOM_STATE)
w_hstat = train_df['expected_claim_amount'].sample(2000, random_state = CONFIG.RANDOM_STATE)

hstat_features = all_features

for f in hstat_features:
    print(f"Calculating 1D PDP for: {f}")
    # Pass a list [f] to the features argument of compute_pd_table
    pdp_1d_cache[f] = compute_pd_table(model, X_hstat, [f], w_hstat)

print("\n--- 2D PDP and H-statistic calculation ---")
hstat_results = []
pairs = list(itertools.combinations(hstat_features, 2))

for f1, f2 in pairs:
    print(f"Processing pair: {f1} x {f2}")

    # 1. Calculate the 2D PDP table
    grid_2d = compute_pd_table(model, X_hstat, [f1, f2], w_hstat)

    # 2. Merge centered 1D PDP values (pd_centered)
    # Merge 1D values for f1
    grid_2d = pd.merge(
        grid_2d,
        pdp_1d_cache[f1][[f1, 'pd_centered']].rename(columns={'pd_centered': 'pd_centered_f1'}),
        on=f1,
        how='left'
    )

    # Merge 1D values for f2
    grid_2d = pd.merge(
        grid_2d,
        pdp_1d_cache[f2][[f2, 'pd_centered']].rename(columns={'pd_centered': 'pd_centered_f2'}),
        on=f2,
        how='left'
    )

    # 3. Calculate interaction residuals (Interaction = 2D - 1D_1 - 1D_2)
    grid_2d['interaction'] = grid_2d['pd_centered'] - grid_2d['pd_centered_f1'] - grid_2d['pd_centered_f2']

    # 4. H-statistic (ratio of weighted sum of square)
    weights = grid_2d['pd_weights'].fillna(0).values
    numerator = np.sum(weights * (grid_2d['interaction'] ** 2))
    denominator = np.sum(weights * (grid_2d['pd_centered'] ** 2))

    if denominator < 1e-10:
        h_stat = 0.0
    else:
        h_stat = np.sqrt(np.clip(numerator / denominator, 0.0, 1.0))

    interaction_size = np.sqrt(numerator / np.sum(weights))

    hstat_results.append({
        'Feature 1': f1,
        'Feature 2': f2,
        'H_statistic': h_stat,
        'Interaction_Size': interaction_size
    })

# Complete the symmetric matrix (diagonal elements and reversed pairs) and convert it to a DataFrame
final_results = []
for row in hstat_results:
    final_results.append({
        'Feature 1': row['Feature 1'],
        'Feature 2': row['Feature 2'],
        'H_statistic': row['H_statistic'],
        'Interaction_Size': row['Interaction_Size']
    })
    final_results.append({
        'Feature 1': row['Feature 2'],
        'Feature 2': row['Feature 1'],
        'H_statistic': row['H_statistic'],
        'Interaction_Size': row['Interaction_Size']
    })

for f in hstat_features:
    final_results.append({
        'Feature 1': f,
        'Feature 2': f,
        'H_statistic': 0.0,
        'Interaction_Size': 0.0
    })

hstat_df = pd.DataFrame(final_results)

# Fix category order so that it can be passed directly to the previous plotting code
hstat_df['Feature 1'] = pd.Categorical(
    hstat_df['Feature 1'].map(FEATURE_DISPLAY_MAP), categories=[FEATURE_DISPLAY_MAP[v] for v in hstat_features]
)
hstat_df['Feature 2'] = pd.Categorical(
    hstat_df['Feature 2'].map(FEATURE_DISPLAY_MAP), categories=[FEATURE_DISPLAY_MAP[v] for v in hstat_features][::-1]
)

In [ ]:
p = ggplot(hstat_df, aes(x='Feature 1', y='Feature 2', fill='Interaction_Size')) + \
    geom_tile(color="white") + \
    geom_text(aes(label='round(Interaction_Size * 100, 2)'), size=10, color="white") + \
    theme_minimal() + \
    theme(
        axis_text_x=element_text(rotation=90, hjust=1),
        figure_size=(7, 6),
        panel_grid=element_blank(),
        legend_position="none"
    ) + \
    scale_fill_continuous(cmap_name='PuBu', limits = (-7/1000, 20/1000)) + \
    labs(x='', y='', fill='Interaction_Size')

p

### Two-way PDP

In [ ]:
features = ("duration_year_band_num", "sum_assured_band")
print(f"\n--- {features[0]}, {features[1]} ---\n")

# Calculate PDP
pdp_results = partial_dependence(
    estimator=model,
    X=X_train,
    features=features,
    sample_weight=train_df['expected_claim_amount'],
    kind='average'
)

pdp_df = pd.DataFrame({
    features[0]: np.repeat(pdp_results['grid_values'][0], len(pdp_results['grid_values'][1])),
    features[1]: np.tile(pdp_results['grid_values'][1], len(pdp_results['grid_values'][0])),
    'p_index': pdp_results['average'][0].flatten()
})


In [ ]:
p = ggplot(pdp_df) + \
    geom_tile(aes(x=features[0], y=features[1], fill='p_index')) + \
    theme_minimal() + \
    theme(
        legend_position="right",
        figure_size=(7, 3),
        legend_key_width=10,
        legend_key_height=160
    ) + \
    scale_fill_continuous('cividis') + \
    labs(
        fill="",
        x=FEATURE_DISPLAY_MAP[features[0]],
        y=FEATURE_DISPLAY_MAP[features[1]]
    )
p.show()


### SHAP

In [ ]:
# Build the SHAP explainer and calculate SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)
shap_values.feature_names = [FEATURE_DISPLAY_MAP[f] for f in shap_values.feature_names]

# Visualize overall drivers of deviation (Summary Plot)
shap.plots.beeswarm(shap_values, max_display=15)
